# Partition 8 Full Streaming EDA

Full partition 8 diagnostics without loading the full parquet into memory.

This notebook streams selected columns in batches, accumulates small aggregate tables, and plots those aggregates. It does not train a model and does not save a cleaned dataset.

## 1. Setup

In [ ]:
from pathlib import Path
import gc

import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

def resolve_partition_path(partition_id, data_dir=DATA_DIR):
    candidates = [
        data_dir / "train.parquet" / f"partition_id={partition_id}",
        data_dir / "train.parquet" / f"partition_id={partition_id}.parquet",
        data_dir / f"partition_id={partition_id}",
        data_dir / f"partition_id={partition_id}.parquet",
        data_dir / f"part_{partition_id}.parquet",
    ]
    existing = [path for path in candidates if path.exists()]
    if not existing:
        checked = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Could not find partition {partition_id}. Checked:\n{checked}")
    return existing[0]

PARTITION8_PATH = resolve_partition_path(8)
print(f"partition 8 path: {PARTITION8_PATH}")

## 2. Schema

In [ ]:
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
ID_COLS = ["date_id", "time_id", "symbol_id"]

dataset = ds.dataset(PARTITION8_PATH, format="parquet")
columns = dataset.schema.names
FEATURE_COLS = [col for col in columns if col.startswith("feature_")]
PRESENT_ID_COLS = [col for col in ID_COLS if col in columns]

print(f"column count: {len(columns)}")
print(f"feature count: {len(FEATURE_COLS)}")
print(f"target exists: {TARGET_COL in columns}")
print(f"weight exists: {WEIGHT_COL in columns}")
print(f"ID columns present: {PRESENT_ID_COLS}")

missing_required = [col for col in [TARGET_COL, WEIGHT_COL] if col not in columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

## 3. Stream full partition and accumulate aggregates

This cell scans the full partition once. It keeps only aggregate vectors and grouped tables in memory.

In [ ]:
BATCH_SIZE = 8_192
SELECTED_COLS = PRESENT_ID_COLS + FEATURE_COLS + [TARGET_COL, WEIGHT_COL]

def weighted_mean(values, weights):
    total_weight = weights.sum()
    if total_weight == 0:
        return np.nan
    return np.average(values, weights=weights)

def stream_full_partition_stats(parquet_path):
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=SELECTED_COLS,
        batch_size=BATCH_SIZE,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    n_rows = 0
    feature_missing_counts = pd.Series(0, index=FEATURE_COLS, dtype="int64")
    missing_count_hist = pd.Series(0, index=range(len(FEATURE_COLS) + 1), dtype="int64")

    sum_y = 0.0
    sum_y2 = 0.0
    sum_w = 0.0
    sum_w2 = 0.0
    sum_yw = 0.0

    feature_sum = pd.Series(0.0, index=FEATURE_COLS, dtype="float64")
    feature_sum2 = pd.Series(0.0, index=FEATURE_COLS, dtype="float64")
    feature_sum_xy = pd.Series(0.0, index=FEATURE_COLS, dtype="float64")
    feature_non_null = pd.Series(0, index=FEATURE_COLS, dtype="int64")

    # Track exact unique values while cardinality remains small. If a column grows
    # past the cap, stop storing the full set and mark it high-cardinality.
    distinct_cap = 1_000
    feature_unique_values = {col: set() for col in FEATURE_COLS}
    feature_unique_over_cap = {col: False for col in FEATURE_COLS}

    missing_indicator_sum = pd.Series(0.0, index=FEATURE_COLS, dtype="float64")
    missing_indicator_sum_y = pd.Series(0.0, index=FEATURE_COLS, dtype="float64")

    group_parts = {"date_id": [], "time_id": [], "symbol_id": []}

    for batch_idx, record_batch in enumerate(scanner.to_batches(), start=1):
        batch_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
        batch_df = batch_df.dropna(subset=[TARGET_COL, WEIGHT_COL])
        if batch_df.empty:
            continue

        y = batch_df[TARGET_COL].astype("float64")
        w = batch_df[WEIGHT_COL].astype("float64")
        X_raw = batch_df[FEATURE_COLS]
        missing_mask = X_raw.isna()
        row_missing_count = missing_mask.sum(axis=1)
        row_missing_rate = row_missing_count / len(FEATURE_COLS)
        X_filled = X_raw.fillna(0).astype("float64")

        rows = len(batch_df)
        n_rows += rows
        feature_missing_counts += missing_mask.sum(axis=0)
        missing_count_hist = missing_count_hist.add(row_missing_count.value_counts(), fill_value=0).astype("int64")

        sum_y += y.sum()
        sum_y2 += np.square(y).sum()
        sum_w += w.sum()
        sum_w2 += np.square(w).sum()
        sum_yw += (y * w).sum()

        feature_sum += X_filled.sum(axis=0)
        feature_sum2 += np.square(X_filled).sum(axis=0)
        feature_sum_xy += X_filled.mul(y, axis=0).sum(axis=0)
        feature_non_null += (~missing_mask).sum(axis=0)

        for col in FEATURE_COLS:
            if feature_unique_over_cap[col]:
                continue
            values = batch_df[col].dropna().unique()
            feature_unique_values[col].update(values.tolist())
            if len(feature_unique_values[col]) > distinct_cap:
                feature_unique_values[col] = set()
                feature_unique_over_cap[col] = True

        missing_indicator_sum += missing_mask.sum(axis=0)
        missing_indicator_sum_y += missing_mask.astype("float64").mul(y, axis=0).sum(axis=0)

        batch_for_groups = batch_df[PRESENT_ID_COLS + [TARGET_COL, WEIGHT_COL]].copy()
        batch_for_groups["abs_target"] = y.abs().values
        batch_for_groups["feature_missing_count"] = row_missing_count.values
        batch_for_groups["feature_missing_rate"] = row_missing_rate.values

        for group_col in group_parts:
            if group_col in batch_for_groups.columns:
                grouped = batch_for_groups.groupby(group_col).agg(
                    rows=(TARGET_COL, "size"),
                    weight_sum=(WEIGHT_COL, "sum"),
                    weight_sum2=(WEIGHT_COL, lambda s: np.square(s).sum()),
                    weight_min=(WEIGHT_COL, "min"),
                    weight_max=(WEIGHT_COL, "max"),
                    weight_mean=(WEIGHT_COL, "mean"),
                    target_sum=(TARGET_COL, "sum"),
                    abs_target_sum=("abs_target", "sum"),
                    missing_count_sum=("feature_missing_count", "sum"),
                    missing_rate_mean=("feature_missing_rate", "mean"),
                )
                group_parts[group_col].append(grouped)

        if batch_idx % 25 == 0:
            print(f"processed batches: {batch_idx:,} | rows: {n_rows:,}")

        del batch_df, y, w, X_raw, X_filled, missing_mask, row_missing_count, row_missing_rate, batch_for_groups
        gc.collect()

    return {
        "n_rows": n_rows,
        "feature_missing_counts": feature_missing_counts,
        "missing_count_hist": missing_count_hist,
        "sum_y": sum_y,
        "sum_y2": sum_y2,
        "sum_w": sum_w,
        "sum_w2": sum_w2,
        "sum_yw": sum_yw,
        "feature_sum": feature_sum,
        "feature_sum2": feature_sum2,
        "feature_sum_xy": feature_sum_xy,
        "feature_non_null": feature_non_null,
        "feature_unique_values": feature_unique_values,
        "feature_unique_over_cap": feature_unique_over_cap,
        "distinct_cap": distinct_cap,
        "missing_indicator_sum": missing_indicator_sum,
        "missing_indicator_sum_y": missing_indicator_sum_y,
        "group_parts": group_parts,
    }

stats = stream_full_partition_stats(PARTITION8_PATH)
print(f"finished rows: {stats['n_rows']:,}")

## 4. Build aggregate result tables

In [ ]:
def combine_group_parts(parts):
    if not parts:
        return pd.DataFrame()
    raw = pd.concat(parts)
    agg_spec = {
        "rows": "sum",
        "weight_sum": "sum",
        "weight_sum2": "sum",
        "weight_min": "min",
        "weight_max": "max",
        "weight_mean": "mean",
        "target_sum": "sum",
        "abs_target_sum": "sum",
        "missing_count_sum": "sum",
        "missing_rate_mean": "mean",
    }
    combined = raw.groupby(level=0).agg(agg_spec)
    combined["mean_weight"] = combined["weight_sum"] / combined["rows"]
    combined["std_weight"] = np.sqrt((combined["weight_sum2"] / combined["rows"] - combined["mean_weight"].pow(2)).clip(lower=0))
    combined["mean_target"] = combined["target_sum"] / combined["rows"]
    combined["mean_abs_target"] = combined["abs_target_sum"] / combined["rows"]
    combined["mean_missing_count"] = combined["missing_count_sum"] / combined["rows"]
    combined["mean_missing_rate"] = combined["mean_missing_count"] / len(FEATURE_COLS)
    return combined.sort_index()

n = stats["n_rows"]
mean_y = stats["sum_y"] / n
mean_w = stats["sum_w"] / n
var_y = stats["sum_y2"] / n - mean_y ** 2
var_w = stats["sum_w2"] / n - mean_w ** 2

feature_mean = stats["feature_sum"] / n
feature_var = stats["feature_sum2"] / n - feature_mean.pow(2)
feature_cov_y = stats["feature_sum_xy"] / n - feature_mean * mean_y
feature_corr_y = feature_cov_y / np.sqrt(feature_var * var_y)
feature_corr_y = feature_corr_y.replace([np.inf, -np.inf], np.nan).dropna()

missing_rate = stats["feature_missing_counts"] / n
missing_indicator_mean = stats["missing_indicator_sum"] / n
missing_indicator_var = missing_indicator_mean * (1 - missing_indicator_mean)
missing_indicator_cov_y = stats["missing_indicator_sum_y"] / n - missing_indicator_mean * mean_y
missing_indicator_corr_y = missing_indicator_cov_y / np.sqrt(missing_indicator_var * var_y)
missing_indicator_corr_y = missing_indicator_corr_y.replace([np.inf, -np.inf], np.nan).dropna()

feature_unique_count = pd.Series({
    col: np.nan if stats["feature_unique_over_cap"][col] else len(stats["feature_unique_values"][col])
    for col in FEATURE_COLS
})

feature_missing_table = pd.DataFrame({
    "missing_count": stats["feature_missing_counts"],
    "missing_rate": missing_rate,
    "non_null_count": stats["feature_non_null"],
    "unique_count_if_le_cap": feature_unique_count,
    "corr_filled0_with_target": feature_corr_y,
    "corr_missing_indicator_with_target": missing_indicator_corr_y,
}).sort_values("missing_rate", ascending=False)

date_stats = combine_group_parts(stats["group_parts"].get("date_id", []))
time_stats = combine_group_parts(stats["group_parts"].get("time_id", []))
symbol_stats = combine_group_parts(stats["group_parts"].get("symbol_id", []))

print(f"rows scanned: {n:,}")
print(f"mean target: {mean_y:.6f}")
print(f"mean weight: {mean_w:.6f}")
display(feature_missing_table.head(20))
display(symbol_stats.sort_values("weight_sum", ascending=False).head(20))

categorical_candidates = feature_missing_table[
    feature_missing_table["unique_count_if_le_cap"].notna()
    & (feature_missing_table["unique_count_if_le_cap"] <= 15)
].sort_values(["unique_count_if_le_cap", "missing_rate"])

print("Categorical candidate features with <= 15 distinct non-null values:")
display(categorical_candidates[["unique_count_if_le_cap", "missing_rate", "corr_filled0_with_target", "corr_missing_indicator_with_target"]])

## 5. Context from existing modeling notebooks

`02_lgbm_baseline.ipynb` and `03_factors_lgbm.ipynb` both drop `feature_09`, `feature_10`, and `feature_11` as discrete/categorical variables. They keep `symbol_id` and `time_id` as model features, build market averages by `date_id`/`time_id`, and build rolling features grouped by `symbol_id`. The LGBM notes also inspect residuals by symbol and call out symbols 4, 12, 17, and 28 as noisier, while symbol-specific residual models overfit.

The checks below therefore focus on low-cardinality features, especially whether any such feature acts like an asset-class bucket related to `symbol_id`, weights, missingness, or target behavior.

In [ ]:
print("Discrete features previously dropped in modeling notebooks: feature_09, feature_10, feature_11")
print("Modeling context: symbol_id and time_id were retained; market averages used date_id/time_id; rolling features used symbol_id.")

candidate_display = categorical_candidates.copy()
candidate_display["unique_values"] = [
    sorted(stats["feature_unique_values"][col]) if not stats["feature_unique_over_cap"][col] else None
    for col in candidate_display.index
]
display(candidate_display[["unique_count_if_le_cap", "unique_values", "missing_rate", "corr_filled0_with_target", "corr_missing_indicator_with_target"]])

plt.figure(figsize=(10, max(4, 0.35 * len(candidate_display))))
if candidate_display.empty:
    plt.text(0.5, 0.5, "No <=15-cardinality feature candidates found", ha="center", va="center")
    plt.axis("off")
else:
    plt.barh(candidate_display.index[::-1], candidate_display["unique_count_if_le_cap"].values[::-1], color="teal")
    plt.xlabel("distinct non-null values")
    plt.title("Low-cardinality feature candidates")
plt.tight_layout()
plt.show()

## 6. Low-cardinality feature relationships to symbols

This second streaming pass reads only candidate categorical features plus IDs, target, and weight. It checks whether candidate values map cleanly to symbols or behave like broader symbol/asset-class buckets.

In [ ]:
def stream_categorical_candidate_stats(parquet_path, categorical_features):
    if not categorical_features:
        return {}, {}

    selected_cols = PRESENT_ID_COLS + categorical_features + [TARGET_COL, WEIGHT_COL]
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=selected_cols,
        batch_size=BATCH_SIZE,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    value_parts = {col: [] for col in categorical_features}
    symbol_value_parts = {col: [] for col in categorical_features}
    time_value_parts = {col: [] for col in categorical_features}

    for batch_idx, record_batch in enumerate(scanner.to_batches(), start=1):
        batch_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
        batch_df = batch_df.dropna(subset=[TARGET_COL, WEIGHT_COL])
        if batch_df.empty:
            continue

        batch_df["abs_target"] = batch_df[TARGET_COL].abs()
        for col in categorical_features:
            value_group = batch_df.groupby(col, dropna=False).agg(
                rows=(TARGET_COL, "size"),
                symbol_count=("symbol_id", "nunique") if "symbol_id" in batch_df.columns else (TARGET_COL, "size"),
                weight_sum=(WEIGHT_COL, "sum"),
                weight_mean=(WEIGHT_COL, "mean"),
                target_sum=(TARGET_COL, "sum"),
                abs_target_sum=("abs_target", "sum"),
            )
            value_parts[col].append(value_group)

            if "symbol_id" in batch_df.columns:
                symbol_value_group = batch_df.groupby(["symbol_id", col], dropna=False).agg(
                    rows=(TARGET_COL, "size"),
                    weight_sum=(WEIGHT_COL, "sum"),
                    abs_target_sum=("abs_target", "sum"),
                )
                symbol_value_parts[col].append(symbol_value_group)

            if "time_id" in batch_df.columns:
                time_value_group = batch_df.groupby(["time_id", col], dropna=False).agg(
                    rows=(TARGET_COL, "size"),
                    weight_sum=(WEIGHT_COL, "sum"),
                )
                time_value_parts[col].append(time_value_group)

        if batch_idx % 50 == 0:
            print(f"categorical pass batches: {batch_idx:,}")

        del batch_df
        gc.collect()

    value_stats = {}
    symbol_value_stats = {}
    time_value_stats = {}

    for col in categorical_features:
        value_stats[col] = pd.concat(value_parts[col]).groupby(level=0).sum() if value_parts[col] else pd.DataFrame()
        if not value_stats[col].empty:
            value_stats[col]["mean_weight"] = value_stats[col]["weight_sum"] / value_stats[col]["rows"]
            value_stats[col]["mean_target"] = value_stats[col]["target_sum"] / value_stats[col]["rows"]
            value_stats[col]["mean_abs_target"] = value_stats[col]["abs_target_sum"] / value_stats[col]["rows"]

        symbol_value_stats[col] = pd.concat(symbol_value_parts[col]).groupby(level=[0, 1]).sum() if symbol_value_parts[col] else pd.DataFrame()
        if not symbol_value_stats[col].empty:
            symbol_value_stats[col]["mean_weight"] = symbol_value_stats[col]["weight_sum"] / symbol_value_stats[col]["rows"]
            symbol_value_stats[col]["mean_abs_target"] = symbol_value_stats[col]["abs_target_sum"] / symbol_value_stats[col]["rows"]

        time_value_stats[col] = pd.concat(time_value_parts[col]).groupby(level=[0, 1]).sum() if time_value_parts[col] else pd.DataFrame()
        if not time_value_stats[col].empty:
            time_value_stats[col]["mean_weight"] = time_value_stats[col]["weight_sum"] / time_value_stats[col]["rows"]

    return value_stats, symbol_value_stats, time_value_stats


categorical_feature_list = categorical_candidates.index.tolist()
value_stats_by_feature, symbol_value_stats_by_feature, time_value_stats_by_feature = stream_categorical_candidate_stats(
    PARTITION8_PATH,
    categorical_feature_list,
)
print(f"categorical candidates analyzed: {categorical_feature_list}")

In [ ]:
for col in categorical_feature_list:
    value_table = value_stats_by_feature[col].sort_values("rows", ascending=False)
    print(f"\n{col}: value-level stats")
    display(value_table)

    symbol_value = symbol_value_stats_by_feature[col]
    if symbol_value.empty:
        continue

    count_matrix = symbol_value["rows"].unstack(fill_value=0)
    row_share_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0)
    max_value_share_by_symbol = row_share_matrix.max(axis=1)

    print(f"{col}: median max value share per symbol = {max_value_share_by_symbol.median():.3f}")
    print(f"{col}: symbols almost entirely one value (>95%) = {(max_value_share_by_symbol > 0.95).sum()} / {len(max_value_share_by_symbol)}")

    plt.figure(figsize=(10, 8))
    sns.heatmap(row_share_matrix, cmap="viridis", cbar_kws={"label": "row share within symbol"})
    plt.title(f"{col}: value distribution within each symbol")
    plt.xlabel(col)
    plt.ylabel("symbol_id")
    plt.tight_layout()
    plt.show()

    if col in time_value_stats_by_feature and not time_value_stats_by_feature[col].empty:
        time_matrix = time_value_stats_by_feature[col]["rows"].unstack(fill_value=0)
        time_share_matrix = time_matrix.div(time_matrix.sum(axis=1), axis=0)
        plt.figure(figsize=(12, 5))
        time_share_matrix.plot(ax=plt.gca(), linewidth=1)
        plt.title(f"{col}: value mix over time_id")
        plt.xlabel("time_id")
        plt.ylabel("row share")
        plt.tight_layout()
        plt.show()

## 7. Full-partition missingness plots

In [ ]:
top_missing = feature_missing_table.sort_values("missing_rate", ascending=False).head(30)

plt.figure(figsize=(10, 8))
plt.barh(top_missing.index[::-1], top_missing["missing_rate"].values[::-1], color="slateblue")
plt.xlabel("missing rate")
plt.title("Full partition 8: top feature missing rates")
plt.tight_layout()
plt.show()

hist = stats["missing_count_hist"]
hist = hist[hist > 0]
plt.figure(figsize=(10, 4))
plt.bar(hist.index, hist.values, color="slateblue")
plt.xlabel("missing feature count per row")
plt.ylabel("rows")
plt.title("Full partition 8: row missingness distribution")
plt.tight_layout()
plt.show()

## 8. Missingness by date, time, and symbol

In [ ]:
for name, table in [("date_id", date_stats), ("time_id", time_stats), ("symbol_id", symbol_stats)]:
    if table.empty:
        continue
    display(table.sort_values("mean_missing_rate", ascending=False).head(15))
    plt.figure(figsize=(12, 4))
    table["mean_missing_rate"].sort_index().plot(color="slateblue")
    plt.title(f"Full partition 8: mean feature missing rate by {name}")
    plt.xlabel(name)
    plt.ylabel("mean missing rate")
    plt.tight_layout()
    plt.show()

## 9. Which features are missing by symbol and over time

This pass reads only the top missing features plus IDs, then builds feature-by-symbol and feature-by-time missing-rate matrices.

In [ ]:
TOP_MISSING_FEATURE_COUNT = 15
TOP_MISSING_FEATURES = feature_missing_table.sort_values("missing_rate", ascending=False).head(TOP_MISSING_FEATURE_COUNT).index.tolist()


def stream_top_missing_feature_group_rates(parquet_path, features):
    selected_cols = PRESENT_ID_COLS + features
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=selected_cols,
        batch_size=BATCH_SIZE,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    group_parts = {"date_id": [], "time_id": [], "symbol_id": []}
    count_parts = {"date_id": [], "time_id": [], "symbol_id": []}

    for batch_idx, record_batch in enumerate(scanner.to_batches(), start=1):
        batch_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
        missing = batch_df[features].isna().astype("int16")

        for group_col in group_parts:
            if group_col not in batch_df.columns:
                continue
            grouped_missing = missing.groupby(batch_df[group_col]).sum()
            grouped_counts = batch_df.groupby(group_col).size().rename("rows")
            group_parts[group_col].append(grouped_missing)
            count_parts[group_col].append(grouped_counts)

        if batch_idx % 50 == 0:
            print(f"top-missing-feature pass batches: {batch_idx:,}")

        del batch_df, missing
        gc.collect()

    rate_tables = {}
    for group_col in group_parts:
        if not group_parts[group_col]:
            rate_tables[group_col] = pd.DataFrame()
            continue
        missing_counts = pd.concat(group_parts[group_col]).groupby(level=0).sum()
        row_counts = pd.concat(count_parts[group_col]).groupby(level=0).sum()
        rate_tables[group_col] = missing_counts.div(row_counts, axis=0)

    return rate_tables


top_missing_feature_group_rates = stream_top_missing_feature_group_rates(PARTITION8_PATH, TOP_MISSING_FEATURES)
print(f"top missing features analyzed: {TOP_MISSING_FEATURES}")

In [ ]:
for group_col, rate_table in top_missing_feature_group_rates.items():
    if rate_table.empty:
        continue

    display(rate_table.head())
    figsize = (12, 8) if group_col != "time_id" else (14, 8)
    plt.figure(figsize=figsize)
    sns.heatmap(rate_table.T, cmap="mako", cbar_kws={"label": "missing rate"})
    plt.title(f"Top missing feature rates by {group_col}")
    plt.xlabel(group_col)
    plt.ylabel("feature")
    plt.tight_layout()
    plt.show()

## 10. Symbol weight and target relationships

In [ ]:
top_symbols = symbol_stats.sort_values("weight_sum", ascending=False).head(25)
display(top_symbols)

fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)
top_symbols["mean_weight"].plot(kind="bar", ax=axes[0], color="darkorange")
axes[0].set_title("Full partition 8: mean weight by top total-weight symbols")
axes[0].set_ylabel("mean weight")

top_symbols["mean_abs_target"].plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Full partition 8: mean absolute responder_6 by same symbols")
axes[1].set_ylabel("mean abs target")

top_symbols["mean_missing_rate"].plot(kind="bar", ax=axes[2], color="slateblue")
axes[2].set_title("Full partition 8: mean missing rate by same symbols")
axes[2].set_ylabel("mean missing rate")
axes[2].set_xlabel("symbol_id")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(symbol_stats["mean_missing_rate"], symbol_stats["mean_weight"], s=np.clip(symbol_stats["rows"], 20, 300), alpha=0.7)
axes[0].set_xlabel("symbol mean feature missing rate")
axes[0].set_ylabel("symbol mean weight")
axes[0].set_title("Do high-missingness symbols carry different weights?")

axes[1].scatter(symbol_stats["mean_weight"], symbol_stats["mean_abs_target"], s=np.clip(symbol_stats["rows"], 20, 300), alpha=0.7)
axes[1].set_xlabel("symbol mean weight")
axes[1].set_ylabel("symbol mean abs responder_6")
axes[1].set_title("Do heavier symbols have larger target moves?")
plt.tight_layout()
plt.show()

display(symbol_stats[["rows", "mean_weight", "mean_abs_target", "mean_missing_rate"]].corr())

## 11. Weight distribution over date and time

These plots use streaming aggregates. They show mean, standard deviation, min, and max weight by `date_id` and `time_id`, plus whether weight changes line up with missingness or target magnitude.

In [ ]:
for name, table in [("date_id", date_stats), ("time_id", time_stats)]:
    if table.empty:
        continue

    fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
    table["mean_weight"].plot(ax=axes[0], color="darkorange", title=f"Mean weight by {name}")
    table["std_weight"].plot(ax=axes[1], color="peru", title=f"Weight std by {name}")
    table["weight_min"].plot(ax=axes[2], color="gray", label="min")
    table["weight_max"].plot(ax=axes[2], color="black", label="max")
    axes[2].set_title(f"Weight min/max by {name}")
    axes[2].legend()
    table["mean_missing_rate"].plot(ax=axes[3], color="slateblue", title=f"Mean missing rate by {name}")
    axes[3].set_xlabel(name)
    plt.tight_layout()
    plt.show()

    print(f"{name}: weight/missingness/target aggregate correlations")
    display(table[["rows", "mean_weight", "std_weight", "mean_abs_target", "mean_missing_rate"]].corr())

## 12. Correlation connections

In [ ]:
top_value_corr = feature_corr_y.reindex(feature_corr_y.abs().sort_values(ascending=False).head(25).index)
top_missing_indicator_corr = missing_indicator_corr_y.reindex(missing_indicator_corr_y.abs().sort_values(ascending=False).head(25).index)

display(top_value_corr.rename("corr_filled0_feature_with_target").to_frame())
display(top_missing_indicator_corr.rename("corr_missing_indicator_with_target").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(top_value_corr.index[::-1], top_value_corr.values[::-1], color=np.where(top_value_corr.values[::-1] >= 0, "steelblue", "firebrick"))
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Full partition 8: feature values vs responder_6")
axes[0].set_xlabel("Pearson correlation")

axes[1].barh(top_missing_indicator_corr.index[::-1], top_missing_indicator_corr.values[::-1], color=np.where(top_missing_indicator_corr.values[::-1] >= 0, "steelblue", "firebrick"))
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Full partition 8: missing indicators vs responder_6")
axes[1].set_xlabel("Pearson correlation")

plt.tight_layout()
plt.show()

## 13. Date-level target and weight behavior

In [ ]:
if not date_stats.empty:
    fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
    date_stats["rows"].plot(ax=axes[0], color="gray", title="rows by date_id")
    date_stats["mean_weight"].plot(ax=axes[1], color="darkorange", title="mean weight by date_id")
    date_stats["mean_target"].plot(ax=axes[2], color="firebrick", title="mean responder_6 by date_id")
    date_stats["mean_missing_rate"].plot(ax=axes[3], color="slateblue", title="mean feature missing rate by date_id")
    axes[3].set_xlabel("date_id")
    plt.tight_layout()
    plt.show()

    display(date_stats[["rows", "mean_weight", "mean_target", "mean_abs_target", "mean_missing_rate"]].corr())